# NB_patient_360 — Silver to Gold

Builds `azuredatabricks_medical.gold.patient_360` — one row per patient, enriched with hospital name, insurance provider name, and aggregated lab-test metrics.

**Load type:** FULL (CREATE OR REPLACE TABLE … AS SELECT)

In [0]:
%sql
CREATE OR REPLACE TABLE azuredatabricks_medical.gold.patient_360 AS
SELECT
    p.patient_id,
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS DATE)                                       AS date_of_birth,
    CAST(FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
    p.gender,
    p.city,
    p.state,
    p.registered_hospital_id                                            AS hospital_id,
    h.hospital_name,
    p.insurance_provider_id,
    ip.provider_name                                                    AS insurance_provider_name,
    COALESCE(lr.total_tests, 0)                                         AS total_tests,
    COALESCE(lr.abnormal_test_count, 0)                                 AS abnormal_test_count,
    COALESCE(lr.critical_test_count, 0)                                 AS critical_test_count,
    lr.last_test_date
FROM azuredatabricks_medical.silver.patients p
LEFT JOIN azuredatabricks_medical.silver.hospitals h
    ON p.registered_hospital_id = h.hospital_id
LEFT JOIN azuredatabricks_medical.silver.insurance_providers ip
    ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN (
    SELECT
        patient_id,
        COUNT(*)                                                        AS total_tests,
        COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)         AS abnormal_test_count,
        COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)         AS critical_test_count,
        MAX(test_date)                                                  AS last_test_date
    FROM azuredatabricks_medical.silver.lab_results
    GROUP BY patient_id
) lr
    ON p.patient_id = lr.patient_id

In [0]:
updated_df = spark.table("azuredatabricks_medical.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))